# 06 — Visualise Results

Exploratory figures for narrative development: spatial posterior means,
demographic effect summaries, and IFS vs AIFS comparison panels.

Once figures stabilise, final versions move to `analysis/figures/` scripts
that write publication-quality PDFs/SVGs directly.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..'))
sys.path.insert(0, os.path.join('..', 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import arviz as az

from nwp_census_eval.db import PipelineDB
from config import SHAPEFILE_PATH

DATA_DIR = 'data'
LEAD = 24

In [ ]:
# IFS vs AIFS: mean bias by lead time
with PipelineDB() as db:
    views = db.registered_views()
    results = {}
    for model in ('ifs', 'aifs'):
        view = f'{model}_bias'
        if view in views:
            results[model] = db.query(f"""
                SELECT lead_time, AVG(bias) AS mean_bias, SQRT(AVG(bias*bias)) AS rmse
                FROM {view}
                GROUP BY lead_time ORDER BY lead_time
            """)

if results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = {'ifs': '#2166ac', 'aifs': '#d6604d'}
    for metric, ax, ylabel in [('mean_bias', axes[0], 'Mean bias (K)'), ('rmse', axes[1], 'RMSE (K)')]:
        for model, df in results.items():
            ax.plot(df['lead_time'], df[metric], label=model.upper(),
                    color=colors[model], marker='o', ms=4)
        ax.axhline(0, color='k', lw=0.8, ls='--')
        ax.set_xlabel('Lead time (h)')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(alpha=0.3)
    fig.suptitle('IFS vs AIFS 2m temperature verification', fontsize=13)
    plt.tight_layout()

In [ ]:
# County choropleth: mean bias at target lead
with PipelineDB() as db:
    df_county = db.query(f"""
        SELECT geo_id, AVG(bias) AS mean_bias
        FROM ifs_bias
        WHERE lead_time = {LEAD}
        GROUP BY geo_id
    """)

gdf = gpd.read_file(SHAPEFILE_PATH)[['GEOID', 'geometry']].rename(columns={'GEOID': 'geo_id'})
gdf = gdf.merge(df_county, on='geo_id', how='left')

fig, ax = plt.subplots(figsize=(14, 7))
gdf.plot(
    column='mean_bias', ax=ax, cmap='RdBu_r', vmin=-2, vmax=2,
    linewidth=0, missing_kwds={'color': 'lightgray'},
    legend=True, legend_kwds={'label': 'Mean bias (K)', 'shrink': 0.5},
)
ax.set_title(f'IFS 2m temperature mean bias | lead {LEAD}h', fontsize=12)
ax.axis('off')
plt.tight_layout()